# Meteorology readiness for the local explorer

Grain: native H3 R5 × UTC date / UTC hour. This notebook inspects the evidence created by `scripts/meteorology_data.py --prepare`. The preparation reads every daily/hourly time and H3 key, verifies source hashes and null counts, and compares three reduction days. Reanalysis consistency does not establish observational accuracy. Run from this checkout or its notebooks folder; no network access or source modification is performed.

In [ ]:
from pathlib import Path
import json, sys
repo = Path.cwd() if (Path.cwd() / 'scripts').is_dir() else Path.cwd().parent
sys.path.insert(0, str(repo / 'scripts'))
from meteorology_data import digest, Store
cache = repo / 'public/local-data/meteorology'
manifest = json.loads((cache / 'catalog.json').read_text())
print(manifest['validation'])
print({'days': len(manifest['dates']), 'cells': len(manifest['cells']), 'daily_records': manifest['source']['products']['daily']['rows'], 'hourly_records': manifest['source']['products']['hourly']['rows']})

## Recheck current file identities
This cell reads about 9 GB locally to recompute the Parquet checksums. Change `source` if the retained products are elsewhere.

In [ ]:
source = repo.parent.parent / 'Data/Meteorology'
assert digest(source / 'catalog.json') == manifest['sourceCatalogSha256']
for name, binding in manifest['bindings'].items():
    assert digest(source / name) == binding['sha256'], name
print('Both source checksums match the validated snapshot.')

In [ ]:
for frequency, columns in manifest['statistics'].items():
    print(frequency)
    for column, statistics in columns.items():
        print(column, statistics)
# Unsupported RH stays null. Source supersaturation is not clipped.
print(manifest['source']['null_policy'])

## Bounded exact queries
One requested metric/day is read. The full hourly history is never brought into memory. Regional hourly means are descriptive unweighted cell means, not area averages.

In [ ]:
store = Store(source, cache)
request = {'snapshot': manifest['snapshot'], 'date': manifest['dates'][-1], 'metric': 'TEMPERATURE_2M_C_MEAN'}
daily = store.query({**request, 'operation': 'daily'})
hourly = store.query({**request, 'operation': 'hourly', 'cell': manifest['cells'][0]})
assert daily['ids'] == manifest['cells']
assert len(hourly['values']) == 24
print('UTC date:', request['date'], 'native daily cells:', len(daily['ids']))
print(list(zip(hourly['hours'], hourly['values'])))